# 🧠 Qwen2-VL-2B-Instruct Streamlit App (Colab)
This notebook launches a **Streamlit web app** inside Google Colab for the Qwen2-VL-2B model.
It can read screenshots, posters, or documents and answer visual questions **without OCR**.

In [ ]:
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers accelerate pillow bitsandbytes streamlit cloudflared


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
!which cloudflared


(Reading database ... 126679 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2025.9.1) over (2025.9.1) ...
Setting up cloudflared (2025.9.1) ...
Processing triggers for man-db (2.10.2-1) ...
/usr/local/bin/cloudflared


In [ ]:
%%writefile app.py
import streamlit as st
from transformers import AutoProcessor, AutoModelForVision2Seq
from PIL import Image
import torch, gc

st.set_page_config(page_title="Qwen2-VL Demo", page_icon="🧠", layout="wide")
st.title("🧠 Qwen2-VL-2B Instruct — Visual Question Answering (OCR-free)")
st.caption("Reads screenshots, posters, and documents directly — no OCR needed.")

device = "cuda" if torch.cuda.is_available() else "cpu"
st.sidebar.success(f"✅ Running on {device.upper()}")

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

@st.cache_resource
def load_model():
    st.info("⏳ Loading Qwen2-VL model (first run can take ~2–5 min on Colab)…")
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID,
        dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
        trust_remote_code=True,
    )
    return processor, model

processor, model = load_model()

uploaded = st.file_uploader("📤 Upload an image", type=["png", "jpg", "jpeg"])
question = st.text_input(
    "💬 Ask a question about the image:",
    "Read all visible text and explain the image in detail."
)

def build_inputs_safe(processor, image, question):
    """
    Build inputs in the most robust way:
    1) Use chat template (official path).
    2) Fallback to <image> placeholder.
    3) Fallback to <image_placeholder>.
    Always wrap image in a list: images=[image]
    """
    # 1) Official chat format
    messages = [
        {"role": "system", "content": "You are a helpful vision-language assistant."},
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question},
            ],
        },
    ]
    try:
        prompt = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        return processor(images=[image], text=prompt, return_tensors="pt")
    except Exception:
        pass

    # 2) Fallback: <image>
    try:
        prompt = (
            "<|im_start|>system\nYou are a helpful vision-language assistant.<|im_end|>\n"
            f"<|im_start|>user\n<image>\n{question}<|im_end|>\n"
            "<|im_start|>assistant\n"
        )
        return processor(images=[image], text=prompt, return_tensors="pt")
    except Exception:
        pass

    # 3) Fallback: <image_placeholder>
    prompt = (
        "<|im_start|>system\nYou are a helpful vision-language assistant.<|im_end|>\n"
        f"<|im_start|>user\n<image_placeholder>\n{question}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )
    return processor(images=[image], text=prompt, return_tensors="pt")

if uploaded:
    image = Image.open(uploaded).convert("RGB")
    st.image(image, caption="Uploaded Image", use_container_width=True)

    if st.button("Ask Qwen-VL"):
        with st.spinner("🤔 Thinking…"):
            try:
                # free any stale CUDA memory
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                inputs = build_inputs_safe(processor, image, question).to(model.device)

                with torch.no_grad():
                    output = model.generate(
                        **inputs,
                        max_new_tokens=768,
                        do_sample=True,
                        temperature=0.7,
                        top_p=0.9,
                    )

                answer = processor.batch_decode(output, skip_special_tokens=True)[0]
                # small cleanup if template tokens leak
                answer = answer.replace("<|im_start|>assistant", "").strip()

                st.subheader("🧩 Answer")
                st.success(answer)

            except Exception as e:
                st.error(f"❌ Model error: {e}\n\n"
                         "Tip: If this persists, restart runtime and run the notebook from the top.")
else:
    st.info("👆 Upload a PNG/JPG and ask a question.")


Overwriting app.py


In [ ]:
!pkill cloudflared


In [ ]:
!pip install -q streamlit cloudflared transformers accelerate pillow torch
!streamlit run app.py &>/dev/null &
!sleep 5
!cloudflared tunnel --url http://localhost:8501


2025-10-13T17:05:58Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-10-13T17:05:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-10-13T17:06:01Z INF +--------------------------------------------------------------------------------------------+
2025-10-13T17:06:01Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-10-13T17:06:01Z INF |  https://maybe-exemption-euros-engines.trycloudflare.c